# UNICEF Intervention Simulator - COMPLETE VERSION

This notebook includes **BOTH**:
- 🔥 **Donor Fatigue**: Penalties for over-communication based on contact frequency
- ⚡ **Severity Penalties**: Automatic corrections for extreme scenarios (scandals, breakthroughs)

## How It Works:
1. Simulates contact history (1-6 contacts in past 3 months)
2. Calculates fatigue penalties based on preferences
3. LLM predicts donor response
4. Applies severity corrections for scandals/breakthroughs
5. Outputs corrected trust/impact for XGBoost NPS calculation in R

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# COMPLETE CODE - WITH FATIGUE + SEVERITY - COPY THIS ENTIRE CELL
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import asyncio
import aiohttp
import json
import os
import time
import random
from typing import Tuple

# Enable asyncio in Jupyter
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    !pip install nest_asyncio --break-system-packages
    import nest_asyncio
    nest_asyncio.apply()

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    "groq_api_key": os.environ.get("GROQ_API_KEY", ""),  # set GROQ_API_KEY in your environment (see .env.example)
    "groq_model": "qwen/qwen3-32b",
    "temperature": 0.0,
    "max_tokens": 1000,
    "timeout_seconds": 45,
    "max_retries": 3,
    "concurrent_limit": 20,
    "input_file": "fragile_donors_with_prompts.csv",
    "output_file": "simulated_intervention_results_complete.csv"
}

# ═══════════════════════════════════════════════════════════════════════════
# INTERVENTION TEXT - CHANGE THIS TO TEST DIFFERENT SCENARIOS
# ═══════════════════════════════════════════════════════════════════════════

# CURRENT: personal thank-you video (detected as EXTREME_POSITIVE).
# Swap this text to test other scenarios (scandal, GDPR notice, ...).
INTERVENTION_TEMPLATE = """Subject: Sarah Wants to Thank You

Dear {donor_name},

Your €50 helped pay for school supplies for 4 children in Rwanda. One of them 
wants to thank you personally.

→ WATCH VIDEO: [unique link to 2-minute personal video]

In the video, you'll meet:
- Sarah, age 9, who received notebooks and pencils funded by your donation
- Her teacher, who explains how your contribution helps
- The classroom where Sarah studies

YOUR EDUCATION IMPACT (your donation priority):
- €17.50 → Sarah's classroom (video shows the result)
  PROOF: Classroom with "Funded by donors like you" sign
  
- €17.50 → School supplies for 3 other children
  PROOF: Inventory records with your donation ID #284
  
- €7.50 → Teachers' salaries and training
  PROOF: Teacher payment records
  
- €7.50 → Educational program coordination
  PROOF: Field coordinator who made this video

100% of your donation supported education programs that align with your priorities.

WHAT HAPPENS NEXT:
- You'll get 3 more video updates this year showing Sarah's progress
- Optional: Send a message back to Sarah (we'll translate and deliver it)
- Dashboard access to track your education projects
- Direct email line to the education coordinator in Rwanda

This is Sarah's education. This is your impact.

With gratitude,
UNICEF Netherlands + Sarah's Teacher, Ms. Uwase

P.S. Sarah drew you a thank-you picture. It's attached.


"""

# ═══════════════════════════════════════════════════════════════════════════
# SYSTEM PROMPT
# ═══════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = """You are a sophisticated simulation engine for computational social science research.

CRITICAL: You MUST produce realistic reactions including EXTREME responses when warranted.

CALIBRATION EXAMPLES:

EXTREME NEGATIVE (Scandal): trust→0.0, impact→0.0, "Never donating again"
NEGATIVE (Spam): trust -0.1, impact 0.0, "Another annoying email"
POSITIVE (Good): trust +0.3, impact +0.3, "Finally some honesty"
EXTREME POSITIVE (Video): trust→1.0, impact→1.0, "I'm crying, this is amazing"

DONOR FATIGUE: If donor was contacted many times and wants LESS communication,
trust may DECREASE even if content is good (communication overload).

Output ONLY valid JSON:
{
  "new_trust": 0.0 or 0.5 or 1.0,
  "new_impact": 0.0 or 0.5 or 1.0,
  "new_alignment_pct": float between 0-100,
  "new_perc_breadth": int 0-8,
  "new_info_spending_more": 0 or 1,
  "qualitative_rationale": "brief explanation (2-3 sentences)"
}"""

# ═══════════════════════════════════════════════════════════════════════════
# DONOR FATIGUE FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def simulate_contact_history(donor_tenure, info_spending_preference):
    """Simulate number of recent contacts from UNICEF in past 3 months."""
    if donor_tenure < 1:
        base_contacts = random.randint(1, 3)
    else:
        base_contacts = random.randint(2, 5)
    
    if info_spending_preference == 1:  # Wants MORE
        n_contacts = base_contacts + random.randint(0, 2)
    elif info_spending_preference == 0:  # Wants LESS
        n_contacts = max(1, base_contacts - random.randint(0, 1))
    else:
        n_contacts = base_contacts
    
    if n_contacts == 1:
        description = "1 email in past 3 months (minimal contact)"
    elif n_contacts == 2:
        description = "2 contacts in past 3 months (low frequency)"
    elif n_contacts == 3:
        description = "3 contacts in past 3 months (moderate frequency)"
    elif n_contacts == 4:
        description = "4 contacts in past 3 months (high frequency)"
    else:
        description = f"{n_contacts} contacts in past 3 months (very high frequency - possible fatigue)"
    
    return n_contacts, description

def calculate_fatigue_adjustment(n_contacts, info_spending_preference, current_trust, current_impact, intervention_severity="NEUTRAL"):
    """
    Calculate fatigue penalty adjustments.
    
    NEW: Reduced penalties for boring/neutral content (privacy emails, etc.)
    """
    trust_adj = 0.0
    impact_adj = 0.0
    rationale = ""
    
    # High contact + wants LESS = significant fatigue
    if info_spending_preference == 0 and n_contacts >= 3:
        trust_adj = -0.2 * (n_contacts - 2)
        impact_adj = -0.1 * (n_contacts - 2)
        rationale = "DONOR FATIGUE: Prefers less communication but contacted frequently"
    
    # Very high contacts (5+) regardless of preference
    elif n_contacts >= 5:
        trust_adj = -0.15 * (n_contacts - 4)
        impact_adj = -0.05 * (n_contacts - 4)
        rationale = "MODERATE FATIGUE: High contact frequency"
    
    # Wants MORE + high contacts = welcomed
    elif info_spending_preference == 1 and n_contacts >= 3:
        trust_adj = 0.05
        impact_adj = 0.05
        rationale = "WELCOMED CONTACT: Donor wants more communication"
    
    # ═══════════════════════════════════════════════════════════════════════
    # NEW: REDUCE FATIGUE FOR BORING/NEUTRAL CONTENT
    # ═══════════════════════════════════════════════════════════════════════
    if intervention_severity in ["NEGATIVE", "NEUTRAL"]:
        # Boring emails (privacy policy, etc.) are easier to ignore
        # Reduce fatigue impact by 60%
        trust_adj = trust_adj * 0.4
        impact_adj = impact_adj * 0.4
        
        if rationale and trust_adj != 0:
            rationale += " (reduced for neutral content)"
    # ═══════════════════════════════════════════════════════════════════════
    
    # Cap adjustments
    trust_adj = max(min(trust_adj, 0.5), -0.5)
    impact_adj = max(min(impact_adj, 0.5), -0.5)
    
    expected_trust = max(0.0, min(1.0, current_trust + trust_adj))
    expected_impact = max(0.0, min(1.0, current_impact + impact_adj))
    
    return trust_adj, impact_adj, expected_trust, expected_impact, rationale

# ═══════════════════════════════════════════════════════════════════════════
# SEVERITY DETECTION FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def assess_intervention_severity(intervention_text: str) -> Tuple[str, float, float]:
    """Detect intervention severity and return penalties/bonuses."""
    text_lower = intervention_text.lower()
    
    scandal_keywords = [
        "scandal", "fraud", "misuse", "bonuses", "luxury travel",
        "executive", "inappropriate", "investigation", "audit found",
        "embezzle", "waste", "corruption", "rooftop lounge"
    ]
    
    breakthrough_keywords = [
        "personal video", "child thanking you", "by name",
        "video message from", "sarah wants to thank"
    ]
    
    strong_proof_keywords = [
        "gps coordinates", "vaccination records", "photos", "proof",
        "dashboard", "track your", "evidence"
    ]
    
    annoying_keywords = [
        "privacy policy", "terms of service", "legal update",
        "compliance", "gdpr", "cookie consent"
    ]
    
    # Check for scandal
    scandal_count = sum(1 for kw in scandal_keywords if kw in text_lower)
    if scandal_count >= 2:
        return "EXTREME_NEGATIVE", -0.8, -0.7
    
    # Check for breakthrough
    breakthrough_count = sum(1 for kw in breakthrough_keywords if kw in text_lower)
    if breakthrough_count >= 1:
        return "EXTREME_POSITIVE", 0.8, 0.8
    
    # Check for strong proof
    proof_count = sum(1 for kw in strong_proof_keywords if kw in text_lower)
    if proof_count >= 3:
        return "VERY_POSITIVE", 0.4, 0.4
    
    # Check for annoying
    annoying_count = sum(1 for kw in annoying_keywords if kw in text_lower)
    if annoying_count >= 2:
        return "NEGATIVE", -0.1, 0.0
    
    return "NEUTRAL", 0.0, 0.0

def apply_severity_correction(
    intervention_text: str,
    llm_trust: float,
    llm_impact: float,
    current_trust: float,
    current_impact: float
) -> Tuple[float, float, str, bool]:
    """Apply severity corrections to LLM predictions."""
    severity, trust_penalty, impact_penalty = assess_intervention_severity(intervention_text)
    
    # Neutral/positive - no correction needed
    if severity in ["NEUTRAL", "POSITIVE"]:
        return llm_trust, llm_impact, severity, False
    
    # Very positive - boost if LLM too conservative
    if severity == "VERY_POSITIVE":
        min_trust_increase = 0.3
        min_impact_increase = 0.3
        
        corrected_trust = llm_trust if llm_trust >= current_trust + min_trust_increase else min(1.0, current_trust + min_trust_increase)
        corrected_impact = llm_impact if llm_impact >= current_impact + min_impact_increase else min(1.0, current_impact + min_impact_increase)
        
        if corrected_trust != llm_trust or corrected_impact != llm_impact:
            return corrected_trust, corrected_impact, severity, True
        return llm_trust, llm_impact, severity, False
    
    # EXTREME NEGATIVE (Scandal) - force severe drops
    if severity == "EXTREME_NEGATIVE":
        corrected_trust = max(0.0, current_trust + trust_penalty)
        corrected_impact = max(0.0, current_impact + impact_penalty)
        
        # If LLM already harsh, don't override
        if llm_trust <= corrected_trust and llm_impact <= corrected_impact:
            return llm_trust, llm_impact, severity, False
        
        return corrected_trust, corrected_impact, severity, True
    
    # EXTREME POSITIVE (Breakthrough) - allow maximal increase
    if severity == "EXTREME_POSITIVE":
        corrected_trust = min(1.0, current_trust + trust_penalty)
        corrected_impact = min(1.0, current_impact + impact_penalty)
        
        if llm_trust >= corrected_trust and llm_impact >= corrected_impact:
            return llm_trust, llm_impact, severity, False
        
        return corrected_trust, corrected_impact, severity, True
    
    # NEGATIVE - small penalty
    if severity == "NEGATIVE":
        corrected_trust = max(0.0, current_trust + trust_penalty)
        corrected_impact = llm_impact
        
        if llm_trust <= corrected_trust:
            return llm_trust, llm_impact, severity, False
        
        return corrected_trust, corrected_impact, severity, True
    
    return llm_trust, llm_impact, severity, False

# ═══════════════════════════════════════════════════════════════════════════
# PROMPT CREATION
# ═══════════════════════════════════════════════════════════════════════════

def create_user_prompt(conditioning_prompt, intervention, current_trust, current_impact, 
                       alignment_pct, info_spending_preference, n_contacts, contact_description,
                       expected_trust, expected_impact, fatigue_rationale):
    """Create user prompt with fatigue and intervention context."""
    alignment_display = alignment_pct * 100 if alignment_pct <= 1.0 else alignment_pct
    
    if info_spending_preference == 1:
        comm_pref = "You WANT MORE communication from UNICEF"
    elif info_spending_preference == 0:
        comm_pref = "You WANT LESS communication from UNICEF"
    else:
        comm_pref = "You have neutral communication preferences"
    
    return f"""{conditioning_prompt}

---

COMMUNICATION HISTORY & DONOR FATIGUE:
UNICEF has contacted you {contact_description}.
{comm_pref}.

{fatigue_rationale if fatigue_rationale else 'Fatigue level: Minimal'}

IMPORTANT: Consider whether this additional contact feels welcome or overwhelming.

---

INTERVENTION SCENARIO:
{intervention}

---

CONTEXT:
- Your current trust: {current_trust:.1f}
- Your current impact: {current_impact:.1f}
- Your alignment: {alignment_display:.0f}%
- Recent contacts: {contact_description}

EXPECTED CHANGES (considering fatigue):
- Trust might change to ~{expected_trust:.1f} (intervention quality + fatigue)
- Impact might change to ~{expected_impact:.1f}

React authentically:
- If scandal: trust should crash
- If excellent: trust can increase significantly
- If fatigued: even good content may annoy

Output ONLY the JSON object."""

# ═══════════════════════════════════════════════════════════════════════════
# API CLIENT
# ═══════════════════════════════════════════════════════════════════════════

class GroqClient:
    def __init__(self, api_key, model):
        self.api_key = api_key
        self.model = model
        self.base_url = "https://api.groq.com/openai/v1/chat/completions"
    
    async def generate(self, session, system_prompt, user_prompt, temperature, max_tokens):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        
        async with session.post(self.base_url, headers=headers, json=payload) as response:
            response.raise_for_status()
            data = await response.json()
            return data["choices"][0]["message"]["content"]

# ═══════════════════════════════════════════════════════════════════════════
# SIMULATION FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

async def process_single_donor(donor_id, row, client, session, semaphore):
    """Process donor with both fatigue and severity corrections."""
    async with semaphore:
        # Extract values
        conditioning_prompt = row['conditioning_prompt']
        alignment_pct = row.get('alignment_pct', 0.94)
        current_trust = row.get('idx_trust', 0.0)
        current_impact = row.get('idx_impact', 0.0)
        donor_tenure = row.get('n_yrs_active', 1.0)
        info_spending_preference = row.get('info_spending_more', 0)
        
        # Calculate fatigue WITH severity context
        n_contacts, contact_description = simulate_contact_history(donor_tenure, info_spending_preference)
        
        # Assess intervention severity first
        severity, _, _ = assess_intervention_severity(INTERVENTION_TEMPLATE)
        
        # Pass severity to fatigue calculation (reduced for boring content)
        trust_adj, impact_adj, expected_trust, expected_impact, fatigue_rationale = calculate_fatigue_adjustment(
            n_contacts, info_spending_preference, current_trust, current_impact, severity
        )
        
        user_prompt = create_user_prompt(
            conditioning_prompt, INTERVENTION_TEMPLATE, current_trust, current_impact,
            alignment_pct, info_spending_preference, n_contacts, contact_description,
            expected_trust, expected_impact, fatigue_rationale
        )
        
        for attempt in range(CONFIG["max_retries"]):
            try:
                raw_response = await asyncio.wait_for(
                    client.generate(session, SYSTEM_PROMPT, user_prompt, 
                                  CONFIG["temperature"], CONFIG["max_tokens"]),
                    timeout=CONFIG["timeout_seconds"]
                )
                
                # Clean response
                clean_response = raw_response.strip()
                if "<think>" in clean_response:
                    clean_response = clean_response.split("</think>")[-1].strip()
                if "```json" in clean_response:
                    clean_response = clean_response.split("```json")[1].split("```")[0].strip()
                elif "```" in clean_response:
                    clean_response = clean_response.split("```")[1].split("```")[0].strip()
                if not clean_response.startswith("{"):
                    start = clean_response.find("{")
                    end = clean_response.rfind("}") + 1
                    if start != -1 and end != 0:
                        clean_response = clean_response[start:end]
                
                response_data = json.loads(clean_response)
                
                required_fields = ["new_trust", "new_impact", "new_alignment_pct", 
                                 "new_perc_breadth", "new_info_spending_more", 
                                 "qualitative_rationale"]
                if not all(f in response_data for f in required_fields):
                    raise ValueError("Missing required fields")
                
                # Get LLM predictions
                llm_trust = response_data["new_trust"]
                llm_impact = response_data["new_impact"]
                
                # APPLY SEVERITY CORRECTION
                corrected_trust, corrected_impact, severity, correction_applied = apply_severity_correction(
                    INTERVENTION_TEMPLATE, llm_trust, llm_impact, current_trust, current_impact
                )
                
                # Override with corrected values
                response_data["new_trust"] = corrected_trust
                response_data["new_impact"] = corrected_impact
                
                # Add metadata
                response_data["donor_id"] = donor_id
                response_data["success"] = True
                response_data["attempts"] = attempt + 1
                
                # Fatigue metadata
                response_data["n_recent_contacts"] = n_contacts
                response_data["contact_description"] = contact_description
                response_data["fatigue_trust_adj"] = trust_adj
                response_data["fatigue_impact_adj"] = impact_adj
                response_data["had_fatigue"] = abs(trust_adj) > 0.1 or abs(impact_adj) > 0.05
                
                # Severity metadata
                response_data["severity_detected"] = severity
                response_data["severity_correction_applied"] = correction_applied
                response_data["llm_trust_original"] = llm_trust
                response_data["llm_impact_original"] = llm_impact
                
                # Deltas
                response_data["delta_trust"] = corrected_trust - current_trust
                response_data["delta_impact"] = corrected_impact - current_impact
                
                # Flag if fatigue negatively affected
                fatigue_impact_detected = (
                    (response_data["delta_trust"] < 0 and trust_adj < 0) or
                    (response_data["delta_impact"] < 0 and impact_adj < 0)
                )
                response_data["fatigue_affected_outcome"] = fatigue_impact_detected
                
                flags = []
                if correction_applied:
                    flags.append("SEVERITY")
                if fatigue_impact_detected:
                    flags.append("FATIGUE")
                flag_str = f" [{'+'.join(flags)}]" if flags else ""
                
                print(f"✓ Donor {donor_id:3d} | Trust: {current_trust:.1f}→{corrected_trust:.1f} | Impact: {current_impact:.1f}→{corrected_impact:.1f} | Contacts: {n_contacts}{flag_str}")
                
                return response_data
                
            except Exception as e:
                if attempt < CONFIG["max_retries"] - 1:
                    print(f"⚠ Donor {donor_id:3d} error (attempt {attempt + 1}): {str(e)[:50]}")
                    await asyncio.sleep(2 ** attempt)
                else:
                    return {
                        "donor_id": donor_id,
                        "new_trust": None,
                        "new_impact": None,
                        "success": False,
                        "attempts": CONFIG["max_retries"],
                        "qualitative_rationale": f"FAILED: {str(e)[:100]}"
                    }

async def run_simulation(df):
    """Run complete simulation."""
    client = GroqClient(CONFIG["groq_api_key"], CONFIG["groq_model"])
    semaphore = asyncio.Semaphore(CONFIG["concurrent_limit"])
    
    severity, _, _ = assess_intervention_severity(INTERVENTION_TEMPLATE)
    
    print(f"\n{'='*70}")
    print(f"🚀 COMPLETE SIMULATOR (FATIGUE + SEVERITY)")
    print(f"{'='*70}")
    print(f"Model:      {CONFIG['groq_model']}")
    print(f"Donors:     {len(df)}")
    print(f"Severity:   {severity}")
    print(f"{'='*70}\n")
    
    start_time = time.time()
    
    timeout = aiohttp.ClientTimeout(total=CONFIG["timeout_seconds"])
    async with aiohttp.ClientSession(timeout=timeout) as session:
        tasks = [process_single_donor(idx, row, client, session, semaphore) 
                for idx, row in df.iterrows()]
        results = await asyncio.gather(*tasks)
    
    elapsed = time.time() - start_time
    results_df = pd.DataFrame(results)
    
    print(f"\n{'='*70}")
    print(f"✅ COMPLETE!")
    print(f"{'='*70}")
    print(f"Success:    {results_df['success'].sum()}/{len(results_df)}")
    print(f"Time:       {elapsed/60:.1f} minutes")
    
    if 'severity_correction_applied' in results_df.columns:
        print(f"Severity corrections: {results_df['severity_correction_applied'].sum()}")
    if 'had_fatigue' in results_df.columns:
        print(f"Fatigue detected:     {results_df['had_fatigue'].sum()}")
        print(f"Fatigue affected:     {results_df['fatigue_affected_outcome'].sum()}")
    
    print(f"{'='*70}\n")
    return results_df

# ═══════════════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════════════

print("📂 Loading donor data...")
df = pd.read_csv(CONFIG['input_file'])
print(f"✓ Loaded {len(df)} donors\n")

print("🔄 Starting simulation...\n")
results_df = await run_simulation(df)

print("💾 Saving results...")
final_df = df.merge(results_df, left_index=True, right_on='donor_id', how='left')
final_df.to_csv(CONFIG['output_file'], index=False)
print(f"✓ Saved to '{CONFIG['output_file']}'\n")

if results_df['success'].sum() > 0:
    successful = results_df[results_df['success'] == True]
    
    print("📊 RESULTS:")
    print(f"  Δ Trust:    {successful['delta_trust'].mean():+.2f}")
    print(f"  Δ Impact:   {successful['delta_impact'].mean():+.2f}")
    print(f"  % Improved: {100 * (successful['delta_trust'] > 0).mean():.1f}%")
    print(f"  % Worsened: {100 * (successful['delta_trust'] < 0).mean():.1f}%")
    
    print(f"\n📝 Sample responses:")
    for i in range(min(3, len(successful))):
        row = successful.iloc[i]
        flags = []
        if row.get('severity_correction_applied', False):
            flags.append("SEVERITY")
        if row.get('fatigue_affected_outcome', False):
            flags.append("FATIGUE")
        flag_str = f" [{'+'.join(flags)}]" if flags else ""
        
        print(f'\n  Donor {i+1}{flag_str}:')
        print(f'    LLM: {row["llm_trust_original"]} → Final: {row["new_trust"]} (trust)')
        print(f'    Contacts: {row["n_recent_contacts"]} | Severity: {row["severity_detected"]}')
        print(f'    "{row["qualitative_rationale"]}"')

print(f"\n{'='*70}")
print("✅ READY FOR R XGBOOST SCORING!")
print(f"{'='*70}")
print(f"Use: {CONFIG['output_file']}")
print("The new_trust and new_impact values include BOTH fatigue and severity corrections!")